# Practice 01 — NumPy & pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/01-numpy-pandas.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/01-numpy-pandas.ipynb)

Companion drills for **Phase 1** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with [vectorization](https://learn-by-visuallization.org/illustrated/1-scientific-stack/numpy-vectorization.html),
[broadcasting](https://learn-by-visuallization.org/illustrated/1-scientific-stack/numpy-broadcasting.html),
[groupby](https://learn-by-visuallization.org/illustrated/1-scientific-stack/pandas-groupby.html) and
[merge](https://learn-by-visuallization.org/illustrated/1-scientific-stack/pandas-merge.html).

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup).

In [ ]:
NEEDED = [("numpy", "numpy"), ("pandas", "pandas")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
import pandas as pd
rng = np.random.default_rng(0)


### Exercise 1 — vectorized z-scores (no loops!)
`X` is 200 rows × 4 feature columns on wildly different scales. Standardize **each column** to
mean 0, std 1 — in one vectorized expression. The check also rejects `for`-loop solutions by
timing nothing: it inspects your result only, but write it vector-style anyway — that habit *is* Phase 1.

*Hint: `(X - X.mean(axis=0)) / X.std(axis=0)` — axis 0 collapses rows, leaving one stat per column.*

In [ ]:
X = rng.normal(loc=[10, 500, -3, 0.01], scale=[2, 120, 0.5, 0.002], size=(200, 4))

Z = None  # TODO: standardized copy of X


In [ ]:
check("Z has X's shape",
      lambda: Z is not None and Z.shape == X.shape, "same (200, 4) array back")
check("every column mean ~ 0",
      lambda: np.allclose(Z.mean(axis=0), 0, atol=1e-9), "subtract X.mean(axis=0)")
check("every column std ~ 1",
      lambda: np.allclose(Z.std(axis=0), 1, atol=1e-9), "divide by X.std(axis=0)")

### Exercise 2 — a distance matrix by broadcasting
`P` is 6 points in 2-D. Build the 6×6 matrix `D` where `D[i, j]` is the **Euclidean distance**
between point i and point j — no loops.

*Hint: `diff = P[:, None, :] - P[None, :, :]` has shape (6, 6, 2); then
`np.sqrt((diff**2).sum(axis=-1))`. This is the (n, 1, d) vs (1, n, d) broadcasting trick.*

In [ ]:
P = rng.uniform(0, 10, size=(6, 2))

D = None  # TODO: pairwise distances via broadcasting


In [ ]:
def _loop_D():
    n = len(P)
    out = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sqrt(((P[i] - P[j]) ** 2).sum())
    return out

check("D matches the loop version",
      lambda: D is not None and np.allclose(D, _loop_D()),
      "P[:, None, :] - P[None, :, :] then square, sum axis=-1, sqrt")
check("diagonal is zero", lambda: np.allclose(np.diag(D), 0), "distance to itself")

### Exercise 3 — boolean masks
From `data`, select `hot`: the rows whose **first column** is above that column's median.
Then compute `frac`: what fraction of ALL rows are hot (a float).

*Hint: `mask = data[:, 0] > np.median(data[:, 0])`, then `data[mask]` and `mask.mean()`.*

In [ ]:
data = rng.normal(size=(100, 3))

hot  = None  # TODO: rows where col 0 > median of col 0
frac = None  # TODO: fraction of rows selected


In [ ]:
_m = data[:, 0] > np.median(data[:, 0])
check("hot has exactly the masked rows",
      lambda: hot is not None and hot.shape == data[_m].shape and np.allclose(hot, data[_m]),
      "index the array with the boolean mask: data[mask]")
check("frac is the mask's mean",
      lambda: frac is not None and abs(frac - _m.mean()) < 1e-12,
      "a boolean mask's .mean() IS the fraction of True")

### Exercise 4 — groupby
`orders` is a tiny order log. Compute `rev`: a Series of **total revenue per category**
(revenue = qty × price), sorted descending.

*Hint: make a `revenue` column first, then `groupby("category")["revenue"].sum().sort_values(ascending=False)`.*

In [ ]:
orders = pd.DataFrame({
    "category": ["books", "toys", "books", "games", "toys", "books"],
    "qty":      [2,       1,      1,       3,       4,      1],
    "price":    [15.0,    30.0,   40.0,    20.0,    10.0,   25.0],
})

rev = None  # TODO: revenue per category, descending


In [ ]:
check("revenue per category is right",
      lambda: rev is not None and dict(rev) == {"books": 95.0, "games": 60.0, "toys": 70.0},
      "books = 2*15 + 1*40 + 1*25 = 95")
check("sorted descending",
      lambda: list(rev.index) == ["books", "toys", "games"],
      ".sort_values(ascending=False)")

### Exercise 5 — merge without losing anyone
Count orders per customer into `per_cust` (a Series indexed by name) — **including Dana, who has
zero orders**. A plain inner merge silently drops her; that silent drop is the #1 real-world join bug.

*Hint: `customers.merge(orders_small, on="cust_id", how="left")`, then group by name and
count a column from the right side (e.g. `order_id`) — count ignores NaN, giving Dana 0.*

In [ ]:
customers = pd.DataFrame({"cust_id": [1, 2, 3, 4],
                          "name": ["Asha", "Bo", "Chen", "Dana"]})
orders_small = pd.DataFrame({"order_id": [10, 11, 12, 13, 14],
                             "cust_id":  [1, 1, 2, 3, 1]})

per_cust = None  # TODO: orders per customer, Dana included with 0


In [ ]:
check("counts are Asha 3, Bo 1, Chen 1, Dana 0",
      lambda: per_cust is not None and
              {k: int(v) for k, v in per_cust.items()} == {"Asha": 3, "Bo": 1, "Chen": 1, "Dana": 0},
      'how="left" keeps Dana; .count() on order_id ignores her NaN')

### Stretch goal — pivot
From `orders`, build a category × (total qty, total revenue) table with `pivot_table` or `agg`.
Compare with the [groupby explainer](https://learn-by-visuallization.org/illustrated/1-scientific-stack/pandas-groupby.html)'s split-apply-combine picture.

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")
